# 04 Abstract Factory (Fabryka Abstrakcyjna) | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: rodziny powiązanych obiektow
2. 🆚 Abstract Factory vs Factory Method
3. 🏗️ Implementacja: hierarchia fabryk
4. 🎨 Przyklad: motywy UI
5. 🌍 Zastosowania i kiedy uzywac

## 1. 🔹 Problem: rodziny powiązanych obiektow

Abstract Factory to wzorzec kreacyjny pozwalajacy tworzyc rodziny
powiazanych obiektow bez podawania ich konkretnych klas.

Problem: aplikacja UI musi dzialac na Windows i macOS. Kazde
srodowisko ma inny wyglad przyciskow, pol tekstowych i dialogow.
Jesli w kodzie tworzymy konkretne klasy (`WindowsButton()`,
`MacButton()`), zmiana platformy wymaga edycji wielu miejsc.

Rozwiazanie: fabryka abstrakcyjna definiuje interfejs tworzenia
calej rodziny obiektow. Konkretna fabryka (np. `WindowsFactory`)
tworzy produkty pasujace do jednej rodziny.

> 💡 Kluczowa roznica: Abstract Factory tworzy RODZINY obiektow,
> Factory Method tworzy JEDEN obiekt. Jezeli masz wiele powiazan
> produktow - uzyj Abstract Factory.

In [ ]:
# Problem: hardkodowane konkretne klasy tworza powiazanie z platforma
class WinButton:
    def render(self): print('[Win Button]')

class MacButton:
    def render(self): print('(Mac Button)')

class WinInput:
    def render(self): print('[Win Input___]')

class MacInput:
    def render(self): print('(Mac Input...)')

# Zla implementacja: konkretne klasy w logice biznesowej
def render_login_bad(platform: str) -> None:
    if platform == 'windows':
        btn = WinButton()
        inp = WinInput()
    else:
        btn = MacButton()
        inp = MacInput()
    inp.render()
    btn.render()
    # dodanie Linux wymaga modyfikacji tej funkcji!

render_login_bad('windows')
render_login_bad('mac')

---

### 🐍 Cwiczenia - problem

1. Policz ile miejsc w `render_login_bad` musisz zmienic, dodajac
   Linux jako trzecia platforme.
2. Wyobraz sobie aplikacje z 10 widgetami (Button, Input, Dialog,
   Label, Menu, ...). Ile `if/else` potrzebujesz dla 3 platform?
3. *(Trudniejsze)* Napisz funkcje `count_changes(widgets, platforms)`
   obliczajaca ile miejsc w kodzie wymaga zmiany przy dodaniu
   nowej platformy przy podejsciu bez wzorca.

In [ ]:
# Cwiczenie 1: zmiana przy dodaniu Linux
modyfikacje = {
    'render_login_bad': 1,   # 1 nowa galaz
    'inne_funkcje': '?',     # kazda funkcja z widgetami
}
print('Miejsc do zmiany:', modyfikacje)

In [ ]:
# Cwiczenie 2: liczba if/else dla 10 widgetow x 3 platformy
widgets = 10
platforms = 3
# kazda funkcja uzywajaca widgetow ma po 1 if/else na platforme
possible_ifs = widgets * (platforms - 1)
print(f'Minimalna liczba galezi if: {possible_ifs}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: kalkulator kosztu zmian
def count_changes(widgets: int, platforms: int) -> dict:
    # Bez wzorca: kazdy widget ma if/elif dla platform
    # Dodanie nowej platformy -> zmiana w kazdym widgecie
    new_platform_cost = widgets
    new_widget_cost = platforms
    return {
        'dodanie_platformy': new_platform_cost,
        'dodanie_widgetu': new_widget_cost,
        'skala': 'O(widgets)',
    }

result = count_changes(widgets=10, platforms=3)
for k, v in result.items():
    print(f'{k}: {v}')

## 2. 🔹 Abstract Factory vs Factory Method

Oba wzorce sa kreacyjne i unikaja hardkodowania konkretnych klas.
Kluczowe roznice:

| Cecha | Factory Method | Abstract Factory |
|---|---|---|
| Tworzy | 1 produkt | rodzine produktow |
| Jak | metoda w klasie | osobny obiekt-fabryka |
| Rozszerzenie | nowa podklasa Creatora | nowa podklasa Factory |
| Uzycie | gdy nie wiadomo jaki typ | gdy produkty sa powiazane |

Factory Method: Logistyka tworzy jeden Transport (Truck lub Ship).
Abstract Factory: UIFactory tworzy cala rodzine widgetow
(Button + Input + Dialog) dla jednej platformy.

> 💡 Jesli twoja fabryka ma wiecej niz jedna metode `create_xxx()` -
> prawdopodobnie potrzebujesz Abstract Factory, nie Factory Method.

In [ ]:
from abc import ABC, abstractmethod

# Factory Method: jeden produkt
class Transport(ABC):
    @abstractmethod
    def deliver(self) -> str: ...

class Logistics(ABC):
    @abstractmethod
    def create_transport(self) -> Transport: ...  # JEDNA metoda

# Abstract Factory: rodzina produktow
class GUIWidget(ABC):
    @abstractmethod
    def render(self) -> str: ...

class GUIFactory(ABC):
    @abstractmethod
    def create_button(self) -> GUIWidget: ...   # WIELE metod
    @abstractmethod
    def create_input(self) -> GUIWidget: ...
    @abstractmethod
    def create_menu(self) -> GUIWidget: ...

print('Factory Method: 1 metoda create_transport()')
print('Abstract Factory: 3 metody create_button/input/menu()')

---

### 🐍 Cwiczenia - roznice

1. Podaj trzy przyklady, gdzie Factory Method wystarcza (jeden produkt),
   i trzy gdzie potrzebna jest Abstract Factory (rodzina produktow).
2. Napisz `DocumentFactory` z jedną metodą `create_document()` -
   to Factory Method. Nastepnie przepisz na `OfficeFactory` tworzacy
   `Document`, `Spreadsheet`, `Presentation` - to Abstract Factory.
3. *(Trudniejsze)* Czy Abstract Factory moze uzywac wewnetrznie
   Factory Method? Napisz przyklad gdzie tak jest.

In [ ]:
# Cwiczenie 1: klasyfikacja
examples = {
    'Factory Method (1 produkt)': [
        'Tworzenie polaczenia DB (jedno polaczenie)',
        'Parsowanie pliku (jeden parser)',
        'Tworzenie powiadomienia (jeden kanal)',
    ],
    'Abstract Factory (rodzina)': [
        'Motywy UI (button + input + dialog)',
        'Klienci API (auth + request + response)',
        'Adaptery baz danych (connection + query + transaction)',
    ]
}
for pattern, ex in examples.items():
    print(f'\n{pattern}:')
    for e in ex:
        print(f'  - {e}')

In [ ]:
# Cwiczenie 2: Factory Method -> Abstract Factory
from abc import ABC, abstractmethod

# Factory Method: 1 produkt
class DocCreator(ABC):
    @abstractmethod
    def create_document(self) -> str: ...

class WordCreator(DocCreator):
    def create_document(self) -> str: return 'Word Document'

# Abstract Factory: rodzina
class OfficeFactory(ABC):
    @abstractmethod
    def create_document(self) -> str: ...
    @abstractmethod
    def create_spreadsheet(self) -> str: ...
    @abstractmethod
    def create_presentation(self) -> str: ...

class MicrosoftOfficeFactory(OfficeFactory):
    def create_document(self) -> str: return 'Word .docx'
    def create_spreadsheet(self) -> str: return 'Excel .xlsx'
    def create_presentation(self) -> str: return 'PowerPoint .pptx'

class LibreOfficeFactory(OfficeFactory):
    def create_document(self) -> str: return 'Writer .odt'
    def create_spreadsheet(self) -> str: return 'Calc .ods'
    def create_presentation(self) -> str: return 'Impress .odp'

for factory in [MicrosoftOfficeFactory(), LibreOfficeFactory()]:
    print(factory.create_document(), factory.create_spreadsheet())

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Abstract Factory uzywajaca Factory Method
from abc import ABC, abstractmethod

class Compression(ABC):
    @abstractmethod
    def compress(self, data: str) -> str: ...

class ZipCompression(Compression):
    def compress(self, data: str) -> str: return f'zip({data})'

class TarCompression(Compression):
    def compress(self, data: str) -> str: return f'tar({data})'

class ArchiveFactory(ABC):
    # Factory Method wbudowany w Abstract Factory
    @abstractmethod
    def create_compressor(self) -> Compression: ...
    @abstractmethod
    def create_header(self) -> str: ...

    def create_archive(self, data: str) -> str:  # uzywa FM wewnetrznie
        header = self.create_header()
        comp = self.create_compressor()
        return f'{header}: {comp.compress(data)}'

class ZipArchiveFactory(ArchiveFactory):
    def create_compressor(self) -> Compression: return ZipCompression()
    def create_header(self) -> str: return 'ZIP/1.0'

print(ZipArchiveFactory().create_archive('myfile.txt'))

## 3. 🔹 Implementacja: hierarchia fabryk

Pelna implementacja Abstract Factory sklada sie z:
- Abstrakcyjnych produktow (interfejsy dla kazdego typu widgetu)
- Konkretnych produktow (implementacje per platforma)
- Abstrakcyjnej fabryki (interfejs tworzenia calej rodziny)
- Konkretnych fabryk (jedna per platforma)

Klient operuje wylacznie na abstrakcyjnej fabryce i abstrakcyjnych
produktach - nie wie nic o konkretnych klasach.

> 💡 Zamiast hierarchii klas mozna uzyc prostego slownika fabryk
> (dict[str, AbstractFactory]) - latwiejsze do konfiguracji.

In [ ]:
from abc import ABC, abstractmethod

# Abstrakcyjne produkty
class Button(ABC):
    @abstractmethod
    def render(self) -> str: ...
    @abstractmethod
    def on_click(self) -> str: ...

class Checkbox(ABC):
    @abstractmethod
    def render(self) -> str: ...
    @abstractmethod
    def toggle(self) -> str: ...

# Konkretne produkty Windows
class WindowsButton(Button):
    def render(self) -> str: return '[WIN_BTN]'
    def on_click(self) -> str: return 'Windows click event'

class WindowsCheckbox(Checkbox):
    def render(self) -> str: return '[x] WIN_CHECK'
    def toggle(self) -> str: return 'Windows checkbox toggled'

# Konkretne produkty macOS
class MacButton(Button):
    def render(self) -> str: return '(MAC_BTN)'
    def on_click(self) -> str: return 'macOS click event'

class MacCheckbox(Checkbox):
    def render(self) -> str: return '(v) MAC_CHECK'
    def toggle(self) -> str: return 'macOS checkbox toggled'

# Abstrakcyjna fabryka
class GUIFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...
    @abstractmethod
    def create_checkbox(self) -> Checkbox: ...

# Konkretne fabryki
class WindowsFactory(GUIFactory):
    def create_button(self) -> Button: return WindowsButton()
    def create_checkbox(self) -> Checkbox: return WindowsCheckbox()

class MacFactory(GUIFactory):
    def create_button(self) -> Button: return MacButton()
    def create_checkbox(self) -> Checkbox: return MacCheckbox()

# Klient: operuje na abstrakcjach
def render_app(factory: GUIFactory) -> None:
    btn = factory.create_button()
    chk = factory.create_checkbox()
    print(f'  Button:   {btn.render()} -> {btn.on_click()}')
    print(f'  Checkbox: {chk.render()} -> {chk.toggle()}')

for platform, factory in [('Windows', WindowsFactory()), ('macOS', MacFactory())]:
    print(f'\n{platform}:')
    render_app(factory)

---

### 🐍 Cwiczenia - hierarchia fabryk

1. Dodaj produkt `Dialog` z metodami `show() -> str` i `close() -> str`.
   Zaimplementuj `WindowsDialog` i `MacDialog`. Rozszerz fabryki.
2. Dodaj `LinuxFactory` tworzaca `LinuxButton`, `LinuxCheckbox`,
   `LinuxDialog`. Upewnij sie ze `render_app()` dziala bez zmian.
3. *(Trudniejsze)* Napisz `get_factory(os_name: str) -> GUIFactory`
   wybierajaca fabryke na podstawie nazwy systemu. Uzyj `platform.system()`.

In [ ]:
# Cwiczenie 1: dodanie Dialog do rodziny
class Dialog(ABC):
    @abstractmethod
    def show(self) -> str: ...
    @abstractmethod
    def close(self) -> str: ...

class WindowsDialog(Dialog):
    def show(self) -> str: return '[WIN_DIALOG opened]'
    def close(self) -> str: return '[WIN_DIALOG closed]'

class MacDialog(Dialog):
    def show(self) -> str: return '(MAC_DIALOG opened)'
    def close(self) -> str: return '(MAC_DIALOG closed)'

# Rozszerz GUIFactory i konkretne fabryki o create_dialog()
print(WindowsDialog().show())
print(MacDialog().show())

In [ ]:
# Cwiczenie 2: LinuxFactory
class LinuxButton(Button):
    def render(self) -> str: return '<GTK_BTN>'
    def on_click(self) -> str: return 'GTK click event'

class LinuxCheckbox(Checkbox):
    def render(self) -> str: return '[_] GTK_CHECK'
    def toggle(self) -> str: return 'GTK checkbox toggled'

class LinuxFactory(GUIFactory):
    def create_button(self) -> Button: ...
    def create_checkbox(self) -> Checkbox: ...

print('\nLinux:')
render_app(LinuxFactory())

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: get_factory z platform.system()
import platform

_FACTORY_MAP: dict[str, GUIFactory] = {
    'Windows': WindowsFactory(),
    'Darwin': MacFactory(),
    'Linux': LinuxFactory(),
}

def get_factory(os_name: str = None) -> GUIFactory:
    # hint: platform.system() zwraca 'Windows', 'Darwin', 'Linux'
    if os_name is None:
        os_name = platform.system()
    if os_name not in _FACTORY_MAP:
        raise ValueError(f'Unknown OS: {os_name}')
    return _FACTORY_MAP[os_name]

current = get_factory()
print(f'\nCurrent OS factory: {type(current).__name__}')
render_app(current)

## 4. 🔹 Przyklad: system baz danych

Inny praktyczny przyklad: rodzina obiektow dla roznych silnikow
baz danych. Kazdy silnik ma swoja implementacje polaczenia,
kursora i transakcji - ale interfejs pozostaje taki sam.

Pozwala to na:
- Przetestowanie kodu z SQLite bez koniecznosci uruchamiania PostgreSQL
- Zmiane silnika bazy przez zmiane jednej linii (konfiguracja fabryki)
- Uzycie tych samych migracji z rozna baza danych

In [ ]:
from abc import ABC, abstractmethod

# Abstrakcyjne produkty
class Connection(ABC):
    @abstractmethod
    def connect(self) -> str: ...
    @abstractmethod
    def close(self) -> str: ...

class Cursor(ABC):
    @abstractmethod
    def execute(self, sql: str) -> list: ...

class Transaction(ABC):
    @abstractmethod
    def begin(self) -> str: ...
    @abstractmethod
    def commit(self) -> str: ...
    @abstractmethod
    def rollback(self) -> str: ...

# SQLite rodzina
class SQLiteConnection(Connection):
    def connect(self) -> str: return 'SQLite: connected (in-memory)'
    def close(self) -> str: return 'SQLite: closed'

class SQLiteCursor(Cursor):
    def execute(self, sql: str) -> list:
        return [{'id': 1, 'name': 'Alice'}]

class SQLiteTransaction(Transaction):
    def begin(self) -> str: return 'SQLite: BEGIN'
    def commit(self) -> str: return 'SQLite: COMMIT'
    def rollback(self) -> str: return 'SQLite: ROLLBACK'

# PostgreSQL rodzina
class PostgresConnection(Connection):
    def connect(self) -> str: return 'Postgres: connected (localhost:5432)'
    def close(self) -> str: return 'Postgres: closed'

class PostgresCursor(Cursor):
    def execute(self, sql: str) -> list:
        return [{'id': 1, 'name': 'Bob'}]

class PostgresTransaction(Transaction):
    def begin(self) -> str: return 'Postgres: BEGIN TRANSACTION'
    def commit(self) -> str: return 'Postgres: COMMIT'
    def rollback(self) -> str: return 'Postgres: ROLLBACK'

# Abstrakcyjna fabryka
class DatabaseFactory(ABC):
    @abstractmethod
    def create_connection(self) -> Connection: ...
    @abstractmethod
    def create_cursor(self) -> Cursor: ...
    @abstractmethod
    def create_transaction(self) -> Transaction: ...

class SQLiteFactory(DatabaseFactory):
    def create_connection(self) -> Connection: return SQLiteConnection()
    def create_cursor(self) -> Cursor: return SQLiteCursor()
    def create_transaction(self) -> Transaction: return SQLiteTransaction()

class PostgresFactory(DatabaseFactory):
    def create_connection(self) -> Connection: return PostgresConnection()
    def create_cursor(self) -> Cursor: return PostgresCursor()
    def create_transaction(self) -> Transaction: return PostgresTransaction()

def run_query(factory: DatabaseFactory, sql: str) -> list:
    conn = factory.create_connection()
    cursor = factory.create_cursor()
    tx = factory.create_transaction()
    print(conn.connect())
    print(tx.begin())
    result = cursor.execute(sql)
    print(tx.commit())
    print(conn.close())
    return result

for factory in [SQLiteFactory(), PostgresFactory()]:
    print(f'\n--- {type(factory).__name__} ---')
    data = run_query(factory, 'SELECT * FROM users')
    print(f'Result: {data}')

---

### 🐍 Cwiczenia - bazy danych

1. Napisz `MockDatabaseFactory` zwracajaca proste in-memory implementacje.
   Uzyj jej do przetestowania `run_query` bez prawdziwej bazy.
2. Dodaj `MySQLFactory` do systemu. Upewnij sie ze `run_query`
   dziala bez modyfikacji.
3. *(Trudniejsze)* Napisz `DatabaseContext` przyjmujacy fabryke
   i implementujacy kontekst menadzer (`__enter__`, `__exit__`)
   automatycznie otwierajacy i zamykajacy polaczenie.

In [ ]:
# Cwiczenie 1: MockDatabaseFactory
class MockConnection(Connection):
    def connect(self) -> str: return 'Mock: connected'
    def close(self) -> str: return 'Mock: closed'

class MockCursor(Cursor):
    def execute(self, sql: str) -> list:
        return [{'test': True}]  # zawsze te same dane

class MockTransaction(Transaction):
    def begin(self) -> str: return 'Mock: begin'
    def commit(self) -> str: return 'Mock: commit'
    def rollback(self) -> str: return 'Mock: rollback'

class MockDatabaseFactory(DatabaseFactory):
    def create_connection(self) -> Connection: ...
    def create_cursor(self) -> Cursor: ...
    def create_transaction(self) -> Transaction: ...

result = run_query(MockDatabaseFactory(), 'SELECT 1')
print(f'Mock result: {result}')

In [ ]:
# Cwiczenie 2: MySQLFactory
class MySQLConnection(Connection):
    def connect(self) -> str: return 'MySQL: connected (localhost:3306)'
    def close(self) -> str: return 'MySQL: closed'

class MySQLCursor(Cursor):
    def execute(self, sql: str) -> list:
        return [{'id': 1, 'name': 'Charlie'}]

class MySQLTransaction(Transaction):
    def begin(self) -> str: return 'MySQL: START TRANSACTION'
    def commit(self) -> str: return 'MySQL: COMMIT'
    def rollback(self) -> str: return 'MySQL: ROLLBACK'

class MySQLFactory(DatabaseFactory):
    def create_connection(self) -> Connection: ...
    def create_cursor(self) -> Cursor: ...
    def create_transaction(self) -> Transaction: ...

print('\n--- MySQLFactory ---')
run_query(MySQLFactory(), 'SELECT * FROM orders')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: DatabaseContext (context manager)
class DatabaseContext:
    def __init__(self, factory: DatabaseFactory):
        ...
    def __enter__(self) -> 'DatabaseContext':
        ...
        return self
    def __exit__(self, exc_type, exc_val, exc_tb) -> None:
        if exc_type:
            print(self._tx.rollback())
        else:
            print(self._tx.commit())
        print(self._conn.close())
    def execute(self, sql: str) -> list:
        return self._cursor.execute(sql)

with DatabaseContext(SQLiteFactory()) as db:
    result = db.execute('SELECT * FROM users')
    print(f'Result: {result}')

## 5. 🔹 Zastosowania i kiedy uzywac

Kiedy uzywac Abstract Factory:
- Produkt musi wspolpracowac z innymi produktami z tej samej rodziny
- System ma byc konfigurowalny co do rodziny produktow
- Chcesz zapewnic ze produkty z jednej rodziny sa zawsze uzywane razem

Kiedy NIE uzywac:
- Masz tylko jeden produkt -> uzyj Factory Method
- Rodziny produktow sa proste i nie powiazane ze soba
- Konfiguracja zamiast polimorfizmu bylaby prostsza

| Wzorzec | Kiedy uzywac |
|---|---|
| Factory Method | 1 zmienny typ produktu |
| Abstract Factory | kilka powizanych typow produktow |

Biblioteka standardowa: `xml.etree.ElementTree` vs `xml.dom.minidom`
- Oba tworza rodziny obiektow XML (Element, Attr, Text, Comment)
- Mozna je uzyc wymiennie przez wspolny interfejs

In [ ]:
from abc import ABC, abstractmethod

# Wzorzec: klient Email/SMS z uwierzytelnianiem
class Authenticator(ABC):
    @abstractmethod
    def authenticate(self, token: str) -> bool: ...

class MessageSender(ABC):
    @abstractmethod
    def send(self, to: str, body: str) -> str: ...

class MessageClientFactory(ABC):
    @abstractmethod
    def create_authenticator(self) -> Authenticator: ...
    @abstractmethod
    def create_sender(self) -> MessageSender: ...

class EmailAuth(Authenticator):
    def authenticate(self, token: str) -> bool:
        return token.startswith('email_')

class EmailSender(MessageSender):
    def send(self, to: str, body: str) -> str:
        return f'Email to {to}: {body}'

class SMSAuth(Authenticator):
    def authenticate(self, token: str) -> bool:
        return token.startswith('sms_')

class SMSSender(MessageSender):
    def send(self, to: str, body: str) -> str:
        return f'SMS to {to}: {body}'

class EmailClientFactory(MessageClientFactory):
    def create_authenticator(self) -> Authenticator: return EmailAuth()
    def create_sender(self) -> MessageSender: return EmailSender()

class SMSClientFactory(MessageClientFactory):
    def create_authenticator(self) -> Authenticator: return SMSAuth()
    def create_sender(self) -> MessageSender: return SMSSender()

def send_message(factory: MessageClientFactory, token: str,
                 to: str, body: str) -> None:
    auth = factory.create_authenticator()
    sender = factory.create_sender()
    if auth.authenticate(token):
        print(sender.send(to, body))
    else:
        print(f'Authentication failed for token: {token}')

send_message(EmailClientFactory(), 'email_abc123', 'alice@x.com', 'Hello!')
send_message(SMSClientFactory(), 'sms_xyz789', '+48123456789', 'Hi!')

---

### 🐍 Cwiczenia - zastosowania

1. Napisz `ReportFactory` tworzaca `Chart` (render_chart()) i
   `Table` (render_table()). Dwie implementacje: PDF i HTML.
2. Napisz `PushClientFactory` dla trzeciego kanalu w `send_message`.
   Uzyj bez modyfikacji funkcji `send_message`.
3. *(Trudniejsze)* Napisz `MultiChannelNotifier` przyjmujacy liste
   fabryk i metode `broadcast(token, to, body)` wysylajaca
   do wszystkich kanalow naraz.

In [ ]:
# Cwiczenie 1: ReportFactory
class Chart(ABC):
    @abstractmethod
    def render_chart(self, data: list) -> str: ...

class Table(ABC):
    @abstractmethod
    def render_table(self, data: list) -> str: ...

class ReportFactory(ABC):
    @abstractmethod
    def create_chart(self) -> Chart: ...
    @abstractmethod
    def create_table(self) -> Table: ...

class PDFChart(Chart):
    def render_chart(self, data: list) -> str: return f'PDF Chart: {data}'

class PDFTable(Table):
    def render_table(self, data: list) -> str: return f'PDF Table: {data}'

class HTMLChart(Chart):
    def render_chart(self, data: list) -> str: return f'<canvas>{data}</canvas>'

class HTMLTable(Table):
    def render_table(self, data: list) -> str: return f'<table>{data}</table>'

class PDFReportFactory(ReportFactory):
    def create_chart(self) -> Chart: return PDFChart()
    def create_table(self) -> Table: return PDFTable()

class HTMLReportFactory(ReportFactory):
    def create_chart(self) -> Chart: ...
    def create_table(self) -> Table: ...

for factory in [PDFReportFactory(), HTMLReportFactory()]:
    data = [10, 20, 30]
    print(factory.create_chart().render_chart(data))
    print(factory.create_table().render_table(data))

In [ ]:
# Cwiczenie 2: PushClientFactory
class PushAuth(Authenticator):
    def authenticate(self, token: str) -> bool: return token.startswith('push_')

class PushSender(MessageSender):
    def send(self, to: str, body: str) -> str: return f'Push to {to}: {body}'

class PushClientFactory(MessageClientFactory):
    def create_authenticator(self) -> Authenticator: ...
    def create_sender(self) -> MessageSender: ...

send_message(PushClientFactory(), 'push_abc', 'device_id_123', 'New message!')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: MultiChannelNotifier
class MultiChannelNotifier:
    def __init__(self, factories: list[MessageClientFactory]):
        ...

    def broadcast(self, tokens: list[str], to: str, body: str) -> None:
        # hint: zip(self._factories, tokens)
        ...

notifier = MultiChannelNotifier([
    EmailClientFactory(),
    SMSClientFactory(),
    PushClientFactory(),
])
notifier.broadcast(
    tokens=['email_aaa', 'sms_bbb', 'push_ccc'],
    to='alice',
    body='Important update!'
)